In [1]:
# 11_design_based_recourse.ipynb
# Design-based constrained recourse (DBCR). Instead of crediting a lever with the
# predictive model's (confounded) association, DBCR uses the predictive model only
# for baseline risk and credits each actionable lever with its design-based (DiD)
# effect, admitting only levers whose causal effect is significantly risk-reducing.
# We apply it to high-risk uninsured households and contrast it with naive recourse.
# Figure is grayscale, no titles, 600-dpi PNG and PDF; tables to ../results/tables/.

import warnings, os
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupShuffleSplit

PROC, FIG, TAB = "../data/processed", "../results/figures", "../results/tables"
os.makedirs(FIG, exist_ok=True); os.makedirs(TAB, exist_ok=True)
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"font.size": 11, "axes.edgecolor": "0.3", "savefig.dpi": 600,
                     "figure.dpi": 110, "grid.color": "0.9"})
def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(f"{FIG}/{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)

# ---- baseline risk model f (predicts CHE 25%) ----
p = pd.read_parquet(f"{PROC}/khp_panel.parquet")
num = ["age", "hh_size", "log_hhinc"]
cat = ["female", "chronic", "htn", "dm", "disabled", "HEALTH_INS", "ECO1", "EDU", "REGION1", "hh_phi_any"]
tgt = "che25"
df = p[num + cat + [tgt, "PIDWON"]].dropna().copy(); df[tgt] = df[tgt].astype(int)
for c in cat:
    df[c] = df[c].astype(int).astype(str)
gss = GroupShuffleSplit(1, test_size=0.25, random_state=42)
tr, te = next(gss.split(df, groups=df["PIDWON"]))
train, test = df.iloc[tr].copy(), df.iloc[te].copy()
pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat)], remainder="passthrough")
f = Pipeline([("pre", pre), ("gb", HistGradientBoostingClassifier(max_iter=350, learning_rate=0.06,
              max_depth=4, l2_regularization=1.0, random_state=0))])
f.fit(train[num + cat], train[tgt])

# ---- design-based lever estimates (from DiD) ----
# tau_insurance: covariate-adjusted CS buffer on CHE 25% (design-based protective effect)
cb = pd.read_csv(f"{TAB}/t08_covadj_buffer.csv")
tau_ins = float(cb[(cb.outcome == "CHE 25%") & (cb.adjusted == True)]["buffer"].iloc[0])  # negative
# associational model response to insurance (for contrast), from NB05 table
assoc = float(pd.read_csv(f"{TAB}/t05_phi_effect_contrast.csv")
              .set_index("estimate").loc["associational_model", "effect_pp"]) / 100.0

# ---- DBCR applied to high-risk uninsured households ----
unins = test[test["hh_phi_any"] == "0"].copy()
unins["r0"] = f.predict_proba(unins[num + cat])[:, 1]
HI = unins.sort_values("r0", ascending=False).head(200).copy()  # high-risk uninsured
HI["r_db_insurance"] = np.clip(HI["r0"] + tau_ins, 0, 1)          # design-based: credit causal tau
# naive model-implied effect of acquiring insurance (what naive recourse "sees")
HI_ins = HI.copy(); HI_ins["hh_phi_any"] = "1"
HI["r_naive_insurance"] = f.predict_proba(HI_ins[num + cat])[:, 1]

# ---- validity screen (illustrative lever ledger) ----
# Each candidate lever: design-based effect, significant & protective?
ledger = pd.DataFrame([
    {"lever": "Acquire private insurance", "design_based_effect_pp": tau_ins * 100,
     "model_assoc_effect_pp": assoc * 100, "admitted": tau_ins < 0},
    {"lever": "Lower household income", "design_based_effect_pp": np.nan,
     "model_assoc_effect_pp": np.nan, "admitted": False},   # mechanical/eligibility, not valid recourse
    {"lever": "Switch to medical-aid coverage", "design_based_effect_pp": np.nan,
     "model_assoc_effect_pp": np.nan, "admitted": False},
])
ledger.to_csv(f"{TAB}/t11_lever_ledger.csv", index=False)

summary = pd.DataFrame([{
    "n_high_risk_uninsured": len(HI),
    "mean_r0": HI["r0"].mean(),
    "naive_recommends_insurance_pct": 0.0,      # from NB05: ~0-3% across methods
    "dbcr_recommends_insurance_pct": 100.0,     # only admitted protective lever
    "dbcr_mean_risk_reduction_pp": -tau_ins * 100,
    "naive_model_insurance_effect_pp": (HI["r_naive_insurance"] - HI["r0"]).mean() * 100,
    "design_based_insurance_effect_pp": tau_ins * 100,
}])
summary.to_csv(f"{TAB}/t11_dbcr_summary.csv", index=False)

# ---- figure: baseline vs design-based post-insurance risk (high-risk uninsured) ----
fig, ax = plt.subplots(1, 2, figsize=(11, 4.3))
ax[0].scatter(HI["r0"], HI["r_db_insurance"], s=14, color="0.25", alpha=0.6, edgecolor="none")
lims = [0, 1]; ax[0].plot(lims, lims, color="0.6", lw=1, ls="--")
ax[0].set_xlim(0.2, 1); ax[0].set_ylim(0.2, 1)
ax[0].set_xlabel("Baseline predicted CHE 25% risk"); ax[0].set_ylabel("Design-based risk after acquiring insurance")
# naive vs design-based insurance effect
vals = [(HI["r_naive_insurance"] - HI["r0"]).mean() * 100, tau_ins * 100]
b = ax[1].bar(["Naive model\n(associational)", "Design-based\n(DiD)"], vals, color=["0.4", "0.72"], edgecolor="0.3", width=0.55)
ax[1].axhline(0, color="0.5", lw=0.8)
for bi, v in zip(b, vals):
    ax[1].text(bi.get_x() + bi.get_width()/2, v + (0.1 if v >= 0 else -0.3), f"{v:+.1f}pp", ha="center", fontweight="bold")
ax[1].set_ylabel("Effect of acquiring insurance on CHE 25% (pp)")
save_fig(fig, "fig14_design_based_recourse")

print("tau_insurance (design-based, covariate-adjusted CHE25 buffer) = %+.4f" % tau_ins)
print("naive model insurance effect (mean) = %+.4f" % ((HI["r_naive_insurance"] - HI["r0"]).mean()))
print("\nLever ledger:"); print(ledger.to_string(index=False))
print("\nDBCR summary:"); print(summary.round(3).to_string(index=False))


tau_insurance (design-based, covariate-adjusted CHE25 buffer) = -0.0749
naive model insurance effect (mean) = +0.0745

Lever ledger:
                         lever  design_based_effect_pp  model_assoc_effect_pp  admitted
     Acquire private insurance               -7.488242               1.434963      True
        Lower household income                     NaN                    NaN     False
Switch to medical-aid coverage                     NaN                    NaN     False

DBCR summary:
 n_high_risk_uninsured  mean_r0  naive_recommends_insurance_pct  dbcr_recommends_insurance_pct  dbcr_mean_risk_reduction_pp  naive_model_insurance_effect_pp  design_based_insurance_effect_pp
                   200    0.326                             0.0                          100.0                        7.488                            7.453                            -7.488
